In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler, Imputer
from pyspark.sql.functions import col,  sum as _sum
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import year, month, dayofweek
from pyspark.sql.functions import mode

storage_account = 'adlsgradproject2'
container       = 'retaildata'  

df = spark.read.option('header', True) \
              .option('inferSchema', True) \
              .csv(f'abfss://{container}@{storage_account}.dfs.core.windows.net/raw')

print(f'Row count: {df.count():,}')   
df.printSchema()


In [30]:
null_counts = df.select([_sum(col(c).isNull().cast('int')).alias(c)
                             for c in df.columns])
display(null_counts)

In [31]:
#Remove nulls in IDs
df = df.dropna(subset=["Customer_ID", "Transaction_ID"])

#Drop Duplicates
df = df.dropDuplicates()

#casting to string
cols = ["Customer_ID", "Transaction_ID","Phone" , "Zipcode"]
for c in cols:
    df = df.withColumn(c, col(c).cast("string"))

#Filling nuls in Customers info with Unknown
string_cols = ['Name', 'Email', 'Phone', 'Address','State', 'Zipcode', 'Country']
fill_dict = {col_name: "Unknown" for col_name in string_cols}
df = df.fillna(fill_dict)

#Fill num columns with medain
numeric_median_cols = ['Amount', 'Total_Amount', 'Total_Purchases', 'Age', 'Ratings']
for col_name in numeric_median_cols:
    median_value = df.approxQuantile(col_name, [0.5], 0.01)[0]
    df = df.fillna({col_name: median_value})

#change date column to Date type
df = df.withColumn("Date", to_date(col("Date"), "dd-MM-yy"))
df = df.orderBy("Date")

#Drop nulls in Dates
df = df.dropna(subset=["Date", "Year" , "Month" , "Time"])

#Fill category columns with mode
cols_to_fill = [
    "Product_Category", "City", "Gender", "Income", "Customer_Segment",
    "Product_Brand", "Product_Type", "Shipping_Method", 
    "Payment_Method", "Order_Status", "Feedback"
]
modes_row = df.select([mode(c).alias(c) for c in cols_to_fill]).first()
modes_dict = modes_row.asDict()
df = df.fillna(modes_dict)

#Rename columns that are similar to any Keywords
df= df.withColumnRenamed("State", "state_")
df= df.withColumnRenamed("Date", "full_date")
df = df.withColumnRenamed("Time", "transaction_time")
df= df.withColumnRenamed("Month", "month_name")
df= df.withColumnRenamed("Year", "Year_name")

#Filtering
df = df.filter(col("Total_Amount") > 0)
df = df.filter((col("Age") >= 18) & (col("Age") <= 100))
df = df.filter((col("Ratings") >= 1) & (col("Ratings") <= 5))

def final_validation(df):
    assert df.filter(col("Customer_ID").isNull()).count() == 0
    assert df.filter(col("Transaction_ID").isNull()).count() == 0
    assert df.filter(col("Total_Amount") <= 0).count() == 0
    assert df.filter(col("Age") < 18).count() == 0
    assert df.filter(col("Ratings") > 5).count() == 0
    
    print("Data validation passed!")

final_validation(df)


In [21]:
null_counts = df.select([_sum(col(c).isNull().cast('int')).alias(c)
                             for c in df.columns])
display(null_counts)

In [32]:
#Writng the final schema manually (usefull for loading)
final_columns = [
    "Transaction_ID",
    "Customer_ID",
    "Name",
    "Email",
    "Phone",
    "Address",
    "City",
    "state_",
    "Zipcode",
    "Country",
    "Age",
    "Gender",
    "Income",
    "Customer_Segment",
    "full_date",
    "Year_name",
    "month_name",
    "transaction_time",
    "Total_Purchases",
    "Amount",
    "Total_Amount",
    "Product_Category",
    "Product_Brand",
    "Product_Type",
    "Feedback",
    "Shipping_Method",
    "Payment_Method",
    "Order_Status",
    "Ratings",
    "products"
]
df = df.select(final_columns)

In [ ]:
# Enrich data 
df_enriched = (df.withColumn('day_of_week', dayofweek('full_date')))

# Silver path
silver_path = f'abfss://{container}@{storage_account}.dfs.core.windows.net/curated'

df_enriched.write \
    .mode('overwrite') \
    .parquet(silver_path)p
    
print('Silver layer written successfully!')

In [34]:
#checking the final schema
test_df = spark.read.parquet(silver_path)

test_df.printSchema()
